In [1]:
%pip install -q torch torchvision pandas scikit-learn pillow matplotlib tensorflow


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import tensorflow as tf


In [3]:
%pip install -q kagglehub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import kagglehub
import shutil
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

DOWNLOAD_PATH = Path(
    kagglehub.dataset_download(
        "paramaggarwal/fashion-product-images-small"
    )
)

DESTINATION = DATA_DIR / "fashion-product-images-small"

if DOWNLOAD_PATH.resolve() != DESTINATION.resolve():
    if DESTINATION.exists():
        shutil.rmtree(DESTINATION)

    shutil.copytree(DOWNLOAD_PATH, DESTINATION)

print("Dataset stored at:", DESTINATION)
print("Dataset files:", list(DESTINATION.iterdir()))

c:\Users\rupam\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 565M/565M [01:47<00:00, 5.53MB/s] 

Extracting files...


Dataset stored at: c:\Users\rupam\OneDrive\Desktop\RetailMind Ai\data\fashion-product-images-small
Dataset files: [WindowsPath('c:/Users/rupam/OneDrive/Desktop/RetailMind Ai/data/fashion-product-images-small/images'), WindowsPath('c:/Users/rupam/OneDrive/Desktop/RetailMind Ai/data/fashion-product-images-small/myntradataset'), WindowsPath('c:/Users/rupam/OneDrive/Desktop/RetailMind Ai/data/fashion-product-images-small/styles.csv')]


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
tf.random.set_seed(SEED)

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_DIR = PROJECT_DIR / "data" / "fashion-product-images-small"
DAY5_MODEL_PATH = PROJECT_DIR / "models" / "day5_cnn.keras"
MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COLUMN = "masterCategory"
MAX_IMAGES_PER_CLASS = 500
IMAGE_SIZE = 224
BATCH_SIZE = 32
HEAD_EPOCHS = 5
FINE_TUNE_EPOCHS = 5
LEARNING_RATE = 0.001
FINE_TUNE_LEARNING_RATE = 0.0001

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


In [ ]:
if not DATASET_DIR.exists():
    candidates = [
        PROJECT_DIR / "data" / "fashion-product-images",
        PROJECT_DIR / "fashion-product-images-small",
        PROJECT_DIR / "data"
    ]
    found = None
    for candidate in candidates:
        if candidate.exists() and list(candidate.rglob("styles.csv")):
            found = candidate
            break
    if found is None:
        raise FileNotFoundError("Place the Fashion Product Images dataset inside the project data folder and update DATASET_DIR.")
    DATASET_DIR = found

csv_files = list(DATASET_DIR.rglob("styles.csv"))
image_folders = [path for path in DATASET_DIR.rglob("images") if path.is_dir()]

if not csv_files:
    raise FileNotFoundError("styles.csv was not found.")
if not image_folders:
    raise FileNotFoundError("The images folder was not found.")

CSV_PATH = csv_files[0]
IMAGE_DIR = image_folders[0]

df = pd.read_csv(CSV_PATH, on_bad_lines="skip")
df.columns = df.columns.str.strip()

if "id" not in df.columns or LABEL_COLUMN not in df.columns:
    raise ValueError(f"Required columns are id and {LABEL_COLUMN}. Available columns: {df.columns.tolist()}")

df["image_path"] = df["id"].astype(str).map(lambda value: IMAGE_DIR / f"{value}.jpg")
df = df[df["image_path"].map(lambda path: path.is_file())].copy()
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(str).str.strip()
df = df[df[LABEL_COLUMN].notna() & (df[LABEL_COLUMN] != "")]

counts = df[LABEL_COLUMN].value_counts()
df = df[df[LABEL_COLUMN].isin(counts[counts >= 3].index)].copy()
df = (
    df.groupby(LABEL_COLUMN, group_keys=False)
      .apply(lambda group: group.sample(min(len(group), MAX_IMAGES_PER_CLASS), random_state=SEED))
      .reset_index(drop=True)
)

class_names = sorted(df[LABEL_COLUMN].unique().tolist())
class_to_idx = {name: index for index, name in enumerate(class_names)}
df["label"] = df[LABEL_COLUMN].map(class_to_idx).astype(int)

print("Usable images:", len(df))
print("Number of classes:", len(class_names))
print("Classes:", class_names)
display(df[LABEL_COLUMN].value_counts())


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class FashionDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        with Image.open(row["image_path"]) as image:
            image = image.convert("RGB")
            image = self.transform(image)
        return image, int(row["label"])

train_dataset = FashionDataset(train_df, train_transform)
val_dataset = FashionDataset(val_df, eval_transform)
test_dataset = FashionDataset(test_df, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))


In [ ]:
weights = ResNet50_Weights.DEFAULT
resnet_model = models.resnet50(weights=weights)

for parameter in resnet_model.parameters():
    parameter.requires_grad = False

number_of_features = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(number_of_features, len(class_names))
resnet_model = resnet_model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    resnet_model.fc.parameters(),
    lr=LEARNING_RATE
)


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    all_labels = []
    all_predictions = []

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)

        all_labels.extend(labels.detach().cpu().numpy())
        all_predictions.extend(predictions.detach().cpu().numpy())

    average_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(all_labels, all_predictions)

    return average_loss, accuracy

history = []

for epoch in range(HEAD_EPOCHS):
    train_loss, train_accuracy = run_epoch(
        resnet_model, train_loader, criterion, optimizer
    )
    val_loss, val_accuracy = run_epoch(
        resnet_model, val_loader, criterion
    )

    history.append({
        "stage": "classifier_head",
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy
    })

    print(
        f"Head Epoch {epoch + 1}/{HEAD_EPOCHS} | "
        f"Train Accuracy: {train_accuracy:.4f} | "
        f"Validation Accuracy: {val_accuracy:.4f}"
    )


In [ ]:
for name, parameter in resnet_model.named_parameters():
    if name.startswith("layer4") or name.startswith("fc"):
        parameter.requires_grad = True

optimizer = torch.optim.Adam(
    filter(lambda parameter: parameter.requires_grad, resnet_model.parameters()),
    lr=FINE_TUNE_LEARNING_RATE
)

best_validation_accuracy = 0.0
best_state = None

for epoch in range(FINE_TUNE_EPOCHS):
    train_loss, train_accuracy = run_epoch(
        resnet_model, train_loader, criterion, optimizer
    )
    val_loss, val_accuracy = run_epoch(
        resnet_model, val_loader, criterion
    )

    history.append({
        "stage": "fine_tuning",
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy
    })

    if val_accuracy > best_validation_accuracy:
        best_validation_accuracy = val_accuracy
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in resnet_model.state_dict().items()
        }

    print(
        f"Fine-tuning Epoch {epoch + 1}/{FINE_TUNE_EPOCHS} | "
        f"Train Accuracy: {train_accuracy:.4f} | "
        f"Validation Accuracy: {val_accuracy:.4f}"
    )

if best_state is not None:
    resnet_model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
display(history_df)


In [ ]:
test_loss, transfer_learning_accuracy = run_epoch(
    resnet_model, test_loader, criterion
)

all_true = []
all_pred = []

resnet_model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = resnet_model(images)
        predictions = outputs.argmax(dim=1).cpu().numpy()
        all_pred.extend(predictions)
        all_true.extend(labels.numpy())

print("ResNet50 test loss:", test_loss)
print("ResNet50 test accuracy:", transfer_learning_accuracy)
print(classification_report(
    all_true,
    all_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))


In [ ]:
day5_accuracy = None

if DAY5_MODEL_PATH.exists():
    day5_model = tf.keras.models.load_model(DAY5_MODEL_PATH)

    output_classes = int(day5_model.output_shape[-1])
    if output_classes != len(class_names):
        print(
            "Day 5 model output classes do not match the current dataset classes. "
            "Day 5 comparison skipped."
        )
    else:
        input_shape = day5_model.input_shape
        target_height = input_shape[1] or IMAGE_SIZE
        target_width = input_shape[2] or IMAGE_SIZE

        day5_true = []
        day5_pred = []

        for _, row in test_df.iterrows():
            try:
                image = tf.keras.utils.load_img(
                    row["image_path"],
                    target_size=(target_height, target_width)
                )
                image_array = tf.keras.utils.img_to_array(image) / 255.0
                probabilities = day5_model.predict(
                    np.expand_dims(image_array, axis=0),
                    verbose=0
                )
                day5_true.append(int(row["label"]))
                day5_pred.append(int(np.argmax(probabilities, axis=1)[0]))
            except Exception:
                continue

        if day5_true:
            day5_accuracy = accuracy_score(day5_true, day5_pred)
            print("Day 5 CNN test accuracy:", day5_accuracy)
        else:
            print("No test images could be evaluated by the Day 5 model.")
else:
    print("Day 5 CNN model not found:", DAY5_MODEL_PATH)
    print("Update DAY5_MODEL_PATH to compare against your saved Day 5 model.")


In [ ]:
comparison_rows = [
    {
        "Model": "ResNet50 Transfer Learning",
        "Test Accuracy": transfer_learning_accuracy
    }
]

if day5_accuracy is not None:
    comparison_rows.append({
        "Model": "Day 5 Custom CNN",
        "Test Accuracy": day5_accuracy
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.sort_values("Test Accuracy", ascending=False).reset_index(drop=True))

comparison_df.set_index("Model")["Test Accuracy"].plot(
    kind="bar",
    figsize=(8, 5),
    ylim=(0, 1),
    title="Day 7 - Day 5 CNN vs ResNet50 Transfer Learning"
)
plt.ylabel("Test Accuracy")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
MODEL_PATH = MODEL_DIR / "day7_resnet50_transfer_learning.pth"
CLASS_NAMES_PATH = MODEL_DIR / "day7_class_names.json"
COMPARISON_PATH = REPORT_DIR / "day7_model_comparison.csv"
HISTORY_PATH = REPORT_DIR / "day7_training_history.csv"

torch.save({
    "model_state_dict": resnet_model.state_dict(),
    "class_names": class_names,
    "image_size": IMAGE_SIZE,
    "architecture": "ResNet50"
}, MODEL_PATH)

with open(CLASS_NAMES_PATH, "w", encoding="utf-8") as file:
    json.dump(class_names, file, indent=4)

comparison_df.to_csv(COMPARISON_PATH, index=False)
history_df.to_csv(HISTORY_PATH, index=False)

print("Saved model:", MODEL_PATH)
print("Saved class names:", CLASS_NAMES_PATH)
print("Saved comparison:", COMPARISON_PATH)
print("Saved training history:", HISTORY_PATH)
